In [40]:
!pip install -qU \
langchain \
langchain-core \
langchain-groq \
langchain-tavily \
langgraph

In [41]:
from google.colab import userdata
import os

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

In [42]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY_3")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [43]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY,
    temperature=0
)

In [44]:
search_tool = TavilySearch(max_results=3)

In [45]:
response = llm.invoke("What is Python?")
print(response.content)

**Python** is a high-level, interpreted programming language that is widely used for various purposes such as:

* Web development
* Data analysis and science
* Artificial intelligence and machine learning
* Automation
* Scientific computing
* Education

Python is known for its:

* **Easy-to-learn syntax**: Simple and intuitive syntax makes it a great language for beginners.
* **Large community**: Python has a vast and active community, which means there are many resources available for learning and troubleshooting.
* **Extensive libraries**: Python has a wide range of libraries and frameworks that make it easy to perform various tasks, such as data analysis, web development, and more.
* **Cross-platform compatibility**: Python can run on multiple operating systems, including Windows, macOS, and Linux.

Some of the key features of Python include:

* **Dynamic typing**: Python is dynamically typed, which means you don't need to declare the data type of a variable before using it.
* **Obj

Create Agent

In [47]:
agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt="""
You are an intelligent AI assistant.

Rules:

1. Answer general questions yourself.

2. Use Tavily Search ONLY if the question asks about:
   • latest news
   • current events
   • today's information
   • live updates
   • recent AI developments

3. Never search the web for basic concepts like:
   - Python
   - LangChain
   - Machine Learning
   - Deep Learning

4. Give clear and concise answers.
"""
)

Test

In [48]:
queries = [
    "What is Python?",
    "What is Machine Learning?",
    "Explain LangChain.",
    "Latest AI news",
    "Latest OpenAI updates"
]

for query in queries:

    print("=" * 80)
    print("USER:")
    print(query)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    print("\nAGENT:\n")
    print(response["messages"][-1].content)

USER:
What is Python?

AGENT:

Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysis, artificial intelligence, and more. It is known for its simplicity, readability, and ease of use, making it a popular choice for beginners and experienced programmers alike. Python is often used for scripting, automating tasks, and building applications, and it has a large and active community of developers who contribute to its ecosystem.
USER:
What is Machine Learning?

AGENT:

Machine learning is a subset of artificial intelligence that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time without being explicitly programmed. It allows systems to automatically improve their performance on a task by learning from experience, rather than relying on manual programming or rule-based systems. Machine learnin

Complete Code (Groq + Structured Tool + create_agent)

| Component        | Purpose                                                        |
| ---------------- | -------------------------------------------------------------- |
| `BaseModel`      | Defines the structured input schema                            |
| `Field`          | Adds descriptions and defaults for each input                  |
| `StructuredTool` | Wraps a Python function as a tool with multiple inputs         |
| `ChatGroq`       | Connects LangChain to the Groq LLM                             |
| `create_agent()` | Creates an agent that can decide when to use the tool          |
| `agent.invoke()` | Sends the user's request to the agent and returns the response |


In [53]:
# ==========================================================
# INSTALL PACKAGES
# ==========================================================

!pip install -qU langchain langchain-core langchain-groq langgraph pydantic

# ==========================================================
# IMPORTS
# ==========================================================

from google.colab import userdata
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent

# ==========================================================
# API KEY
# ==========================================================

GROQ_API_KEY = userdata.get("GROQ_API_KEY_3")

# ==========================================================
# LLM
# ==========================================================

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY,
    temperature=0
)

# ==========================================================
# INPUT SCHEMA
# ==========================================================

class TitleInput(BaseModel):
    topic: str = Field(description="Topic of the title")
    tone: str = Field(
        default="friendly",
        description="Tone of the title"
    )

# ==========================================================
# TOOL FUNCTION
# ==========================================================

def title_tool_fn(
    topic: str,
    tone: str = "friendly"
) -> str:
    return f"{tone.title()} Title: {topic}"

# ==========================================================
# STRUCTURED TOOL
# ==========================================================

title_tool = StructuredTool.from_function(
    func=title_tool_fn,
    name="TitleMaker",
    description="Create a title from a topic and tone.",
    args_schema=TitleInput,
)

# ==========================================================
# TEST TOOL
# ==========================================================

print(title_tool.invoke({
    "topic":"LangGraph Tutorials",
    "tone":"friendly"
}))

# ==========================================================
# AGENT
# ==========================================================

agent = create_agent(
    model=llm,
    tools=[title_tool],
    system_prompt="""
You are an AI assistant.

If the user asks for a title,
ALWAYS call the TitleMaker tool.

Never answer directly.
"""
)

# ==========================================================
# TEST QUESTIONS
# ==========================================================

queries = [
    "Generate a friendly title about LangGraph tutorials.",
    "Create a professional title about Machine Learning.",
    "Create a funny title about Python Programming."
]

# ==========================================================
# RUN AGENT FOR ALL QUERIES
# ==========================================================

for i, query in enumerate(queries, start=1):

    print("\n" + "=" * 70)
    print(f"QUESTION {i}")
    print("=" * 70)
    print("USER:")
    print(query)

    try:
        response = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": query
                    }
                ]
            }
        )

        print("\nAGENT:")
        print(response["messages"][-1].content)

    except Exception as e:
        print("\nERROR:")
        print(e)

print("\n" + "=" * 70)
print("ALL QUESTIONS COMPLETED")
print("=" * 70)


Friendly Title: LangGraph Tutorials

QUESTION 1
USER:
Generate a friendly title about LangGraph tutorials.

ERROR:
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kyc6z610ftnr9w52a9k9f4hs` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99738, Requested 338. Please try again in 1m5.664s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

QUESTION 2
USER:
Create a professional title about Machine Learning.


KeyboardInterrupt: 